In [24]:
# ============================================
# Phase 3: Logistic Regression + Anomaly Detection
# ============================================

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve, classification_report

# Load the cleaned dataset from Phase 1 (now fixed)
df = pd.read_csv('../data/telco_cleaned_final.csv')

print(df.shape)
print(df['Churn'].value_counts())

(7043, 22)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [26]:
# Select features for modeling
features = ['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 'PaymentMethod']
target = 'Churn'

X = df[features]
y = df[target]

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, columns=['Contract', 'InternetService', 'PaymentMethod'], drop_first=True)

print(X_encoded.shape)
X_encoded.head()

(7043, 9)


,tenure,MonthlyCharges,Contract_One year,Contract_Two year,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,29.85,False,False,False,False,False,True,False
1,34,56.95,True,False,False,False,False,False,True
2,2,53.85,False,False,False,False,False,False,True
3,45,42.30,True,False,False,False,False,False,False
4,2,70.70,False,False,True,False,False,True,False


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

Train shape: (5634, 9)
Test shape: (1409, 9)
Train churn rate: 0.2653532126375577
Test churn rate: 0.2654364797728886


In [28]:
# Fit logistic regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# Predict on test set
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]  # probability of churn

print("Model fitted successfully")

Model fitted successfully


In [29]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7885024840312278
Precision: 0.62751677852349
Recall: 0.5
F1 Score: 0.5565476190476191
ROC-AUC: 0.8325854452452918

Confusion Matrix:
[[924 111]
 [187 187]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [30]:
# Build a results dataframe aligned with the test set
results = X_test.copy()
results['Actual'] = y_test.values
results['Predicted'] = y_pred
results['Predicted_Proba'] = y_pred_proba

# False negatives: actual churners the model missed
false_negatives = results[(results['Actual'] == 1) & (results['Predicted'] == 0)]
print("False negatives (missed churners):", false_negatives.shape[0])

# False positives: predicted churn but customer stayed
false_positives = results[(results['Actual'] == 0) & (results['Predicted'] == 1)]
print("False positives (false alarms):", false_positives.shape[0])

false_negatives.describe()

False negatives (missed churners): 187
False positives (false alarms): 111


,tenure,MonthlyCharges,Actual,Predicted,Predicted_Proba
count,187.000000,187.000000,187.0,187.0,187.000000
mean,23.128342,61.393583,1.0,0.0,0.294819
std,22.576384,29.464048,0.0,0.0,0.134139
min,1.000000,18.950000,1.0,0.0,0.015536
25%,2.000000,40.400000,1.0,0.0,0.181374
50%,13.000000,55.250000,1.0,0.0,0.329558
75%,41.500000,87.525000,1.0,0.0,0.399778
max,72.000000,115.650000,1.0,0.0,0.499131


In [32]:
'''
Step 1: Threshold Tuning
Right now, log_reg.predict() uses a default 0.5 cutoff. 
Let's see what happens if we lower it, to catch more of those borderline churners we just found.
'''
# Try a few different thresholds and compare precision/recall tradeoff
thresholds = [0.5, 0.4, 0.35, 0.3]

for t in thresholds:
    y_pred_t = (y_pred_proba >= t).astype(int)
    print(f"\nThreshold = {t}")
    print("Precision:", precision_score(y_test, y_pred_t))
    print("Recall:", recall_score(y_test, y_pred_t))
    print("F1 Score:", f1_score(y_test, y_pred_t))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_t))


Threshold = 0.5
Precision: 0.62751677852349
Recall: 0.5
F1 Score: 0.5565476190476191
Confusion Matrix:
[[924 111]
 [187 187]]

Threshold = 0.4
Precision: 0.5655339805825242
Recall: 0.6229946524064172
F1 Score: 0.5928753180661578
Confusion Matrix:
[[856 179]
 [141 233]]

Threshold = 0.35
Precision: 0.5635593220338984
Recall: 0.7112299465240641
F1 Score: 0.6288416075650118
Confusion Matrix:
[[829 206]
 [108 266]]

Threshold = 0.3
Precision: 0.51985559566787
Recall: 0.7700534759358288
F1 Score: 0.6206896551724138
Confusion Matrix:
[[769 266]
 [ 86 288]]


## Threshold Tuning

The default 0.5 classification threshold only catches 50% of actual churners.
Lowering the threshold to 0.35 improves recall to 71% (catching significantly
more at-risk customers) while keeping precision reasonably balanced, achieving
the best F1 score (0.629). Given that the cost of losing a customer typically
outweighs the cost of an unnecessary retention offer, 0.35 is a more
business-appropriate threshold than the default 0.5.

In [33]:
'''
Step 2: Anomaly Detection (after threshold tuning)
Once we settle on a threshold, we'll run Isolation Forest on the full feature set to flag 
customers whose behavior is statistically unusual — this is separate from the 
classification task and adds a complementary "outlier detection" angle to project.
'''
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['Anomaly'] = iso_forest.fit_predict(X_encoded)

# -1 = anomaly, 1 = normal
print(df['Anomaly'].value_counts())

Anomaly
 1    6690
-1     353
Name: count, dtype: int64


In [34]:
anomaly_profile = df.groupby('Anomaly')[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].mean()
print(anomaly_profile)

            tenure  MonthlyCharges  TotalCharges     Churn
Anomaly                                                   
-1       41.966006       51.045892   2535.075779  0.053824
 1       31.864873       65.485411   2272.075336  0.276532


**Key finding:** 
anomalous customers churn far less than normal customers — 5.4% vs 27.7%, roughly a 5x difference. This is the opposite of what people often assume ("anomalies = risky customers"). Here, Isolation Forest is actually flagging customers with an unusual combination of higher tenure + lower monthly charges — and these turn out to be your most loyal, stable segment, not high-risk ones.

**Why this happens:**
Isolation Forest finds points that are easy to isolate based on how rare their feature combination is — it has no concept of "good" or "bad," only "unusual." In this dataset, the most common pattern is shorter-tenure, higher-paying customers (which also happens to be your higher-churn group, consistent with everything you found in Phase 2). So the "unusual" customers — long-tenure, lower-paying — stand out simply because that combination is statistically rarer, not because they're risky.

**a valuable, nuanced insight for project**
 but it reframes how you should present anomaly detection. Rather than "anomaly detection finds risky customers," the honest and more sophisticated framing is:

## Anomaly Detection (Isolation Forest)

Isolation Forest flagged 353 customers (5%) as statistically anomalous based on
tenure, monthly charges, and total charges. Interestingly, these anomalous
customers show a *lower* churn rate (5.4%) than typical customers (27.7%).

This suggests the model is identifying long-tenure, lower-spend customers as
"anomalous" simply because this combination is statistically uncommon in the
dataset — most customers fall into a more common pattern of shorter tenure and
higher spend (which is also the higher-churn group identified in Phase 2).

**Conclusion:** anomaly detection here is better understood as identifying
"unusual but loyal" customers rather than "high-risk" customers. This
demonstrates that statistical outliers don't always correspond to business risk —
an important distinction when applying unsupervised techniques to churn analysis.

In [35]:
# Run Isolation Forest specifically on the false negatives (missed churners)
# Using the same features as the model: tenure, MonthlyCharges, and the encoded columns

fn_features = false_negatives[X_encoded.columns]

iso_forest_fn = IsolationForest(contamination=0.1, random_state=42)
fn_anomaly_labels = iso_forest_fn.fit_predict(fn_features)

false_negatives = false_negatives.copy()
false_negatives['FN_Anomaly'] = fn_anomaly_labels

print(false_negatives['FN_Anomaly'].value_counts())

# Compare anomalous vs normal within the false negatives group
fn_anomaly_profile = false_negatives.groupby('FN_Anomaly')[['tenure', 'MonthlyCharges', 'Predicted_Proba']].mean()
print(fn_anomaly_profile)

FN_Anomaly
 1    168
-1     19
Name: count, dtype: int64
               tenure  MonthlyCharges  Predicted_Proba
FN_Anomaly                                            
-1          43.736842       79.192105         0.108992
 1          20.797619       59.380655         0.315835


## Anomaly Detection on Misclassified Cases

Running Isolation Forest specifically on the model's false negatives (187 missed
churners) revealed two distinct sub-groups:

- **Typical misses (168 customers):** borderline cases with moderate tenure and
  charges, predicted churn probability averaging 0.32 — close to the decision
  threshold. These are largely addressed by the threshold tuning in the
  previous section.

- **Anomalous misses (19 customers):** long-tenure (43.7 months avg), high-spend
  ($79.19/month avg) customers the model was confident would stay (avg predicted
  probability of only 0.11), yet who churned anyway.

**Key insight:** the anomalous group represents a genuine blind spot in the
model — customers with a loyal, high-value profile who left despite no clear
signal in tenure, charges, contract type, internet service, or payment method.
This suggests churn for these customers is likely driven by factors outside
the current feature set (e.g., service issues, competitor offers, or pricing
changes), which the model cannot detect with the data available.

In [36]:
coefficients = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coefficients)

                                 Feature  Coefficient
4            InternetService_Fiber optic     0.962036
7         PaymentMethod_Electronic check     0.512920
8             PaymentMethod_Mailed check     0.021025
1                         MonthlyCharges     0.004530
0                                 tenure    -0.030315
6  PaymentMethod_Credit card (automatic)    -0.040307
5                     InternetService_No    -0.785529
2                      Contract_One year    -0.824419
3                      Contract_Two year    -1.565621


**How to read this:** 
each coefficient shows the effect of that feature relative to its baseline category (remember, drop_first=True dropped Month-to-month, DSL, and Bank transfer as baselines). So:

**Two-year contracts reduce churn risk the most dramatically —**
 even more than EDA alone suggested, this is your single strongest lever

**Fiber optic internet is the strongest single driver of churn —** confirms what you found in EDA, and now you have a precise magnitude, not just "fiber churns more"
**Electronic check payment meaningfully increases churn risk —**
second-strongest driver, again matching your earlier observation
**MonthlyCharges itself has a tiny coefficient (0.0045) —** this is interesting and worth noting: once you control for contract type and internet service, the raw dollar amount barely matters on its own. The earlier correlation of +0.19 you found was likely driven by fiber/contract type rather than price itself. This is a subtle, sophisticated point — multivariate models often reveal that a univariate correlation was actually a proxy for something else.

## Model Coefficients

The logistic regression confirms and quantifies the EDA findings:

- **Two-year contracts** are the strongest retention factor (coefficient: -1.57),
  followed by one-year contracts (-0.82) — both dramatically reduce churn risk
  relative to month-to-month.
- **Fiber optic internet** is the strongest churn driver (coefficient: +0.96).
- **Electronic check payment** is the second-strongest churn driver (+0.51).
- **MonthlyCharges alone has a near-zero effect (+0.005)** once contract type
  and internet service are accounted for — suggesting the earlier univariate
  correlation between charges and churn was largely a proxy for these other
  factors rather than price sensitivity itself.

## Phase 3 Summary

This phase built a logistic regression model to predict churn using EDA-validated
features (tenure, MonthlyCharges, Contract, InternetService, PaymentMethod):

- **Baseline performance:** 78.9% accuracy, 0.83 ROC-AUC
- **Threshold tuning:** lowering the decision threshold from 0.5 to 0.35 improved
  recall from 50% to 71%, better aligning the model with real-world retention
  priorities
- **Anomaly detection on misclassifications** revealed a blind spot: 19 long-tenure,
  high-spend customers churned despite the model confidently predicting they'd stay,
  suggesting churn drivers outside the current feature set (e.g., service quality,
  competitor offers)
- **Model coefficients** confirmed and quantified EDA findings, with contract
  length and internet service type as the dominant churn drivers

These results directly inform Phase 4 (Power BI dashboard) and Phase 6
(Streamlit churn prediction app).